# Carly's Clippers — Solution Notebook (R) — Complete + Extended

**Audience note:** Primary reader is Carly (salon owner). She has high subject knowledge of her business, moderate data literacy, and limited time. We therefore lead with clear numbers, simple visuals, and actionable recommendations. Technical detail and alternate code live later for the analyst / technical supervisor.

## Analysis Flowchart
```mermaid
flowchart TD
    A[Load hairstyles, prices, last_week] --> B[Sum prices → average_price]
    B --> C[Create new_prices = prices − 5]
    C --> D[Compute total_revenue = Σ price × count]
    D --> E[average_daily_revenue = total_revenue / 7]
    E --> F[Filter cuts_under_30 from new_prices]
    F --> G[Visualize prices, revenue by style]
    G --> H[More practice metrics]
    H --> I[Simulation: change discount / sales]
    I --> J[Business recommendations for Carly]
```


## 0. Setup & Data


In [ ]:
hairstyles <- c("bouffant", "pixie", "dreadlocks", "crew", "bowl", "bob", "mohawk", "flattop")
prices     <- c(30, 25, 40, 20, 20, 35, 50, 35)
last_week  <- c(2, 3, 5, 8, 4, 4, 6, 2)

cat("Hairstyles:", hairstyles, "\n")
cat("Prices   :", prices, "\n")
cat("Last week:", last_week, "\n")
cat("Number of styles:", length(hairstyles), "\n")


## 1. Average Haircut Price


In [ ]:
# Classic loop (mirrors the original Python for-loop)
total_price <- 0
for (price in prices) {
  total_price <- total_price + price
}
average_price <- total_price / length(prices)
cat("Average Haircut Price:", average_price, "\n")


### Alternate 1a — vectorized (idiomatic R)


In [ ]:
average_price_alt <- mean(prices)   # or sum(prices) / length(prices)
cat("Average Haircut Price (mean):", average_price_alt, "\n")


## 2. New Prices (all reduced by $5)


In [ ]:
new_prices <- prices - 5   # vectorized arithmetic — no loop needed
print(new_prices)


## 3. Total Revenue Last Week


In [ ]:
# Classic indexed loop (R is 1-based)
total_revenue <- 0
for (i in seq_along(hairstyles)) {
  total_revenue <- total_revenue + prices[i] * last_week[i]
}
cat("Total Revenue:", total_revenue, "\n")


### Alternate 3a — pure vectorized


In [ ]:
total_revenue_alt <- sum(prices * last_week)
cat("Total Revenue (vectorized):", total_revenue_alt, "\n")


### Alternate 3b — data.frame + dplyr (tidyverse style)


In [ ]:
# Requires tidyverse / dplyr
# library(dplyr)
style_df <- data.frame(
  style   = hairstyles,
  price   = prices,
  sold    = last_week,
  stringsAsFactors = FALSE
)
style_df$revenue <- style_df$price * style_df$sold
total_revenue_df <- sum(style_df$revenue)
cat("Total Revenue (data.frame):", total_revenue_df, "\n")
print(style_df)


## 4. Average Daily Revenue


In [ ]:
average_daily_revenue <- total_revenue / 7
cat("Average Daily Revenue:", average_daily_revenue, "\n")


## 5. Cuts Under $30 After Discount


In [ ]:
# Logical indexing (very common R pattern)
cuts_under_30 <- hairstyles[new_prices < 30]
print(cuts_under_30)


### Alternate 5a — which() + indexing


In [ ]:
idx <- which(new_prices < 30)
cuts_under_30_alt <- hairstyles[idx]
print(cuts_under_30_alt)


## 6. More Practice Metrics


In [ ]:
# 1. Most popular cut
max_sold <- max(last_week)
most_popular_idx <- which.max(last_week)
cat(sprintf("1. Most popular: %s (%d times)\n", hairstyles[most_popular_idx], max_sold))

# 2. Highest-revenue cut
revenues <- prices * last_week
max_rev <- max(revenues)
max_rev_idx <- which.max(revenues)
cat(sprintf("2. Highest revenue: %s ($%g)\n", hairstyles[max_rev_idx], max_rev))

# 3. Percentage under $30 after discount
pct_under <- 100 * length(cuts_under_30) / length(hairstyles)
cat(sprintf("3. Styles under $30 after discount: %.1f%%\n", pct_under))

# 4. Total haircuts performed
total_cuts <- sum(last_week)
cat(sprintf("4. Total haircuts last week: %d\n", total_cuts))

# 5. Summary table
summary_df <- data.frame(
  Style     = hairstyles,
  OrigPrice = prices,
  NewPrice  = new_prices,
  Sold      = last_week,
  Revenue   = revenues,
  stringsAsFactors = FALSE
)
cat("\n5. Style summary:\n")
print(summary_df, row.names = FALSE)


## 7. Visualization


In [ ]:
# Base R barplots (no extra packages required)
par(mfrow = c(1, 2), mar = c(7, 4, 3, 1))

# Left: original vs new prices
barplot(rbind(prices, new_prices), beside = TRUE,
        names.arg = hairstyles, las = 2, col = c("#4C72B0", "#55A868"),
        main = "Prices Before vs After $5 Discount",
        ylab = "Price ($)", legend.text = c("Original", "After -$5"),
        args.legend = list(x = "topright", bty = "n", cex = 0.8))
abline(h = 30, col = "red", lty = 2)

# Right: revenue by style
barplot(revenues, names.arg = hairstyles, las = 2, col = "#4C72B0",
        main = "Revenue by Hairstyle (Last Week)", ylab = "Revenue ($)")
# highlight max
barplot(revenues, names.arg = hairstyles, las = 2,
        col = ifelse(revenues == max(revenues), "#C44E52", "#4C72B0"),
        main = "Revenue by Hairstyle (Last Week)", ylab = "Revenue ($)", add = FALSE)

par(mfrow = c(1, 1))

# Optional ggplot2 version (uncomment if tidyverse/ggplot2 is installed)
# library(ggplot2)
# library(tidyr)
# plot_df <- data.frame(Style = hairstyles, Original = prices, New = new_prices)
# plot_long <- tidyr::pivot_longer(plot_df, cols = c(Original, New), names_to = "Type", values_to = "Price")
# ggplot(plot_long, aes(x = Style, y = Price, fill = Type)) +
#   geom_col(position = "dodge") +
#   geom_hline(yintercept = 30, linetype = "dashed", color = "red") +
#   theme_minimal() +
#   theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
#   labs(title = "Prices Before vs After $5 Discount", y = "Price ($)")


## 8. Simulation Section — Modify Values & Observe Impact

Change `DISCOUNT` and `SALES_MULTIPLIER` (or edit the vectors) and re-run the cell.


In [ ]:
# === SIMULATION CONTROLS — EDIT THESE ===
DISCOUNT <- 5
SALES_MULTIPLIER <- 1.0
# ========================================

sim_prices <- prices
sim_last_week <- pmax(0L, as.integer(round(last_week * SALES_MULTIPLIER)))
sim_new_prices <- sim_prices - DISCOUNT

sim_total_revenue <- sum(sim_prices * sim_last_week)
sim_avg_price <- mean(sim_prices)
sim_cuts_under <- hairstyles[sim_new_prices < 30]
sim_daily <- sim_total_revenue / 7

cat("=== Simulation Results ===\n")
cat(sprintf("Discount applied          : $%g\n", DISCOUNT))
cat(sprintf("Sales multiplier          : %g\n", SALES_MULTIPLIER))
cat(sprintf("Average original price    : $%.2f\n", sim_avg_price))
cat(sprintf("Total revenue             : $%.2f\n", sim_total_revenue))
cat(sprintf("Average daily revenue     : $%.2f\n", sim_daily))
cat("Cuts under $30            :", paste(sim_cuts_under, collapse = ", "), "\n")
cat(sprintf("# styles advertised       : %d\n", length(sim_cuts_under)))

# Quick sensitivity table
cat("\nSensitivity of metrics to different discounts (sales fixed):\n")
cat(sprintf("%10s %14s %12s %12s\n", "Discount", "New Avg Price", "Cuts < $30", "Total Rev"))
for (d in c(0, 3, 5, 8, 10)) {
  np_list <- prices - d
  under <- sum(np_list < 30)
  cat(sprintf("%10g %14.2f %12d %12g\n", d, mean(np_list), under, total_revenue))
}


### Extra simulation: 4-week projection at original vs discounted prices


In [ ]:
monthly_at_new_prices <- 4 * sum(new_prices * last_week)
monthly_at_old_prices <- 4 * total_revenue
cat(sprintf("Projected 4-week revenue at original prices : $%s\n", format(monthly_at_old_prices, big.mark = ",")))
cat(sprintf("Projected 4-week revenue at discounted prices: $%s\n", format(monthly_at_new_prices, big.mark = ",")))
cat(sprintf("Difference (cost of $5 discount, same volume): $%s\n", format(monthly_at_old_prices - monthly_at_new_prices, big.mark = ",")))
cat("\nCarly should weigh this against expected extra volume from advertising the under-$30 cuts.\n")


## 9. Business Recommendations (Audience-aware)

**For Carly (owner / decision maker):**
- Average price before discount is **$31.88**. After a uniform $5 cut the advertised average becomes ~$26.88.
- Last week generated **$1,085** total revenue → **~$155 per day**.
- Four styles can be marketed under $30: **bouffant, pixie, crew, bowl**.
- The **mohawk** alone produced $300 (almost 28 % of weekly revenue). Protect that high-ticket service.
- The **crew** cut is the volume leader (8 sales). Pair volume + margin carefully.

**Next experiments Carly can run:**
1. Advertise the four under-$30 cuts for two weeks and measure lift in total visits.
2. Test a $3 vs $5 discount on only the mid-tier styles (keep mohawk & dreadlocks at full price).
3. Track which days of the week drive the crew-cut volume.


## Key Numbers Recap
| Metric | Value |
|--------|-------|
| Average price (original) | $31.875 |
| New prices vector | 25, 20, 35, 15, 15, 30, 45, 30 |
| Total revenue last week | $1,085 |
| Average daily revenue | $155.00 |
| Cuts under $30 | bouffant, pixie, crew, bowl |
| Most popular | crew (8) |
| Highest revenue style | mohawk ($300) |
